# ML-2: Feature Engineering & Reusable Preprocessing Utilities

This notebook defines the domain feature engineering logic and reusable scikit-learn preprocessing utilities for the Predictive Maintenance pipeline.

## Key Objectives:
1. **Domain Feature Construction**:
   - `temperature_difference = Process temperature [K] - Air temperature [K]`
   - `mechanical_power_W = Torque [Nm] * Rotational speed [rpm] * (2 * pi / 60)`
   - `overstrain_index = Tool wear [min] * Torque [Nm]`
2. **Backend API Compatibility**:
   - Supports both dataset column names and backend snake_case keys (`type`, `air_temperature`, `process_temperature`, `rotational_speed`, `torque`, `tool_wear`).
3. **Data Leakage & Target Preservation**:
   - Excludes `UDI`, `Product ID`, `TWF`, `HDF`, `PWF`, `OSF`, `RNF` from predictive features.
   - Keeps `Machine failure` as target.
4. **Reusable Preprocessing Transformer**:
   - `OneHotEncoder(handle_unknown='ignore')` for `Type` categorical variant.
   - `StandardScaler()` for numerical features.
   - Preserves feature names for SHAP explainability in ML-4.
5. **SMOTE & Cross-Validation Integration Design**:
   - Does **not** perform permanent train/test splits or apply SMOTE in ML-2.
   - Provides clean unfitted pipeline utilities so ML-3 can safely incorporate preprocessors and SMOTE within cross-validation folds.

In [1]:
import os
import sys
import numpy as np
import pandas as pd

# Append src module path
sys.path.append(os.path.abspath("../src"))
from feature_engineering import (
    DomainFeatureEngineer, 
    build_preprocessing_pipeline, 
    build_full_feature_pipeline,
    BASE_FEATURE_COLS,
    TARGET_COL,
    EXCLUDED_COLS
)

## 1. Ingest Clean Data & Inspect Feature Sets

In [3]:
data_path = "../../data/cleaned_predictive_maintenance.csv"
if not os.path.exists(data_path):
    data_path = "../../data/ai4i2020.csv"

df = pd.read_csv(data_path)
print(f"Loaded clean dataset shape: {df.shape}")

feature_cols = [c for c in df.columns if c not in EXCLUDED_COLS and c != TARGET_COL]
print(f"Predictive Base Features ({len(feature_cols)}): {feature_cols}")
print(f"Target Column: {TARGET_COL}")

Loaded clean dataset shape: (10000, 7)
Predictive Base Features (6): ['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
Target Column: Machine failure


## 2. Test Domain Feature Engineer & Backend Key Mapping

In [4]:
engineer = DomainFeatureEngineer()
df_engineered = engineer.transform(df[feature_cols])
print(f"Engineered DataFrame Shape: {df_engineered.shape}")
print("New Physical Domain Features Added:")
for col in ['temperature_difference', 'mechanical_power_W', 'overstrain_index']:
    print(f"  • {col}: mean = {df_engineered[col].mean():.4f}")

# Test Backend API snake_case input compatibility
sample_api_input = pd.DataFrame([{
    "type": "M",
    "air_temperature": 298.5,
    "process_temperature": 308.6,
    "rotational_speed": 1550,
    "torque": 42.3,
    "tool_wear": 120
}])
sample_eng = engineer.transform(sample_api_input)
print("\nSample Backend API Input Engineered Successfully:")
print(sample_eng[['Type', 'temperature_difference', 'mechanical_power_W', 'overstrain_index']])

Engineered DataFrame Shape: (10000, 9)
New Physical Domain Features Added:
  • temperature_difference: mean = 10.0006
  • mechanical_power_W: mean = 6279.7450
  • overstrain_index: mean = 4314.6646

Sample Backend API Input Engineered Successfully:
  Type  temperature_difference  mechanical_power_W  overstrain_index
0    M                    10.1         6865.950744            5076.0


## 3. Build Reusable ColumnTransformer Preprocessing Pipeline

In [5]:
pipeline = build_full_feature_pipeline()
X_proc = pipeline.fit_transform(df[feature_cols])
feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()

print(f"Preprocessed Feature Matrix Shape: {X_proc.shape}")
print("\nRecoverable Processed Feature Names (Required for SHAP Explainability):")
for name in feature_names:
    print(f"  • {name}")

Preprocessed Feature Matrix Shape: (10000, 11)

Recoverable Processed Feature Names (Required for SHAP Explainability):
  • Type_H
  • Type_L
  • Type_M
  • Air temperature [K]
  • Process temperature [K]
  • Rotational speed [rpm]
  • Torque [Nm]
  • Tool wear [min]
  • temperature_difference
  • mechanical_power_W
  • overstrain_index


## 4. Reusable Export Utilities for ML-3

ML-3 can import `DomainFeatureEngineer` and `build_preprocessing_pipeline` from `src.feature_engineering` to fit preprocessors inside cross-validation splits and imbalanced-learn pipelines with SMOTE.